<a href="https://colab.research.google.com/github/Raman676/Intaligen_Task/blob/main/Intaligen_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Intaligen Task**
 ~ Raman Dubey

The Task requires us to color the given image of several rice grains and detect edges of these grains for sucessful coloring and provide it on a black background
Using CNNs for the task requires training with some images, so we can use basic CV tools or pre trained models


  Description of the Task

Input

Scanned image of rice grains on light background
Resolution: ~1024×847 pixels
RGB color format

Output

Segmented image with each grain colored distinctly
Black background (no grains)
Instance-level separation of touching grains

  I have used 2 different pipelines

*   Classical Pipeline for more creative and engineered approach
*   Using pretrained semantic models for high accuracy output



**IMPORTANT**



---




Please upload the InputImage.jpg to the cloud environment before running the cells

---



# Using CV tools (Classical pipeline)

  This approach implements a classical computer vision pipeline for rice grain instance segmentation using OpenCV. The approach combines preprocessing, Otsu thresholding, distance transform, and watershed segmentation to separate individual grains from the scanned image. Post-processing operations such as erosion and Gaussian smoothing are applied to improve boundary quality and generate a clean colorized output similar to the provided reference image.

In [ ]:
import cv2
import numpy as np

class RiceSegmentationPipeline:
    """
    A classical computer vision pipeline for instance segmentation of rice grains.
    Focuses on Watershed segmentation and post-process edge smoothing.
    """

    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path
        self.image = None

    def load_image(self):
        print("image loaded")
        self.image = cv2.imread(self.input_path)
        if self.image is None:
            raise ValueError(f"Check the input path")
        return self.image

    def preprocess_and_threshold(self):
        print("Preprocessing")
        gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)

        # CLAHE for handling uneven scanner lighting
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)

        # Otsu's Thresholding
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Clean up noise
        kernel = np.ones((3,3), np.uint8)
        opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
        return opened

    def generate_watershed_markers(self, binary_mask):
        print("applying distance transform")
        kernel = np.ones((3,3), np.uint8)
        sure_bg = cv2.dilate(binary_mask, kernel, iterations=3)

        # Distance transform to find grain centers
        dist_transform = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)

        # 0.3 multiplier: High enough to separate most clumps, low enough to not delete grains
        _, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)

        unknown = cv2.subtract(sure_bg, sure_fg)

        _, markers = cv2.connectedComponents(sure_fg)
        markers = markers + 1
        markers[unknown == 255] = 0

        return markers

    def segment_and_stylize(self, markers):
        print("Applying Watershed")
        markers = cv2.watershed(self.image, markers)

        output_canvas = np.zeros_like(self.image)
        unique_labels = np.unique(markers)
        np.random.seed(42) # For reproducible colors

        # Elliptical kernel for smooth shrinking
        shrink_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

        for label in unique_labels:
            if label == -1 or label == 1:
                continue # Skip boundaries and background

            # Isolate the grain
            mask = np.uint8(markers == label) * 255

            # Filter microscopic noise
            if cv2.countNonZero(mask) < 25:
                continue

            # Stylization: Erode slightly to separate touching borders, then smooth
            mask = cv2.erode(mask, shrink_kernel, iterations=1)
            mask = cv2.GaussianBlur(mask, (5, 5), 0)
            _, final_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

            if cv2.countNonZero(final_mask) == 0:
                continue

            # Apply vibrant random color
            color = np.random.randint(50, 255, size=3).tolist()
            output_canvas[final_mask == 255] = color

        return output_canvas

    def run(self):
        print("Starting Pipeline")
        self.load_image()
        binary_mask = self.preprocess_and_threshold()
        markers = self.generate_watershed_markers(binary_mask)
        final_output = self.segment_and_stylize(markers)

        cv2.imwrite(self.output_path, final_output)
        print(f"Image saved to {self.output_path}")

if __name__ == "__main__":
    pipeline = RiceSegmentationPipeline("InputImage.jpg", "Final_version_Output.jpeg")
    pipeline.run()

# AI Approach

  This approach implements an AI-based rice grain instance segmentation pipeline using the pretrained FastSAM model from Ultralytics. The model detects individual rice grain instances from the scanned image, after which post-processing techniques such as erosion, Gaussian smoothing, and mask refinement are applied to generate clean segmented outputs with stylized colorization similar to the provided reference image.

In [ ]:
import cv2
import numpy as np
import random
!pip install ultralytics
from ultralytics import FastSAM
# we are using the pretrained model
def segment_with_ai(input_path, output_path):
    print("model loaded")
    # This will automatically download the pre-trained weights
    model = FastSAM('FastSAM-s.pt')

    img = cv2.imread(input_path)
    if img is None:
        raise ValueError("recheck image path")

    # Run the model.
    results = model(input_path, device='cpu', retina_masks=True, conf=0.25, iou=0.8)

    # Create the black canvas on which the grains will be
    output_canvas = np.zeros_like(img)
    random.seed(42)

    # Extract the masks generated by the AI
    if results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        print(f" AI detected {len(masks)} distinct instances.")

        # Elliptical kernel to stylize and smooth the edges exactly like the reference
        shrink_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

        for mask in masks:
            # Resize the mask to match the original image dimensions perfectly
            mask_resized = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

            # Convert to uint8 (0 to 255)
            mask_uint8 = (mask_resized * 255).astype(np.uint8)

            # Filter microscopic AI hallucinations
            if cv2.countNonZero(mask_uint8) < 50:
                continue

            # Stylize the mask to get that smooth look
            eroded = cv2.erode(mask_uint8, shrink_kernel, iterations=1)
            smoothed = cv2.GaussianBlur(eroded, (7, 7), 0)
            _, final_mask = cv2.threshold(smoothed, 127, 255, cv2.THRESH_BINARY)

            # Apply vibrant colors
            color = [random.randint(50, 255) for _ in range(3)]
            output_canvas[final_mask == 255] = color

    cv2.imwrite(output_path, output_canvas)
    print(f" AI Segmentation saved to {output_path}")

if __name__ == "__main__":
    segment_with_ai("InputImage.jpg", "OutputImage_AI.jpeg")

   I have used both approachs to showcase my idea which I used in the classical CV model, while we can can also use the new pretrained AI models to get very accurate outputs